# Optimization techniques

Almost every model in this guide is, underneath, an **optimization problem**:
"find the parameters that minimize a loss." It's worth pulling that machinery
into the open, and being precise about two things people often conflate:

- **Model training** learns *parameters* from data — e.g. the slope and
  intercept in [linear regression](../02-regression/linear-regression.ipynb),
  found by minimizing squared error. This happens automatically when you call
  `.fit()`.
- **Hyperparameter tuning** chooses *values you set before training* — e.g. a
  tree's max depth, or *k* in k-means. That's the [next
  notebook](hyperparameter-search.ipynb).

This notebook shows the first kind directly, with
[`argmin`](https://docs.rs/argmin), a general-purpose numerical optimization
crate — so the `.fit()` in earlier chapters stops being a black box.

## Minimizing a loss by hand

We define a toy loss surface with a known minimum at `(3, -1)` — think of it as
a stand-in for a model's loss as a function of its parameters. In `argmin`, a
problem is a type implementing `CostFunction`:

In [ ]:
:dep argmin = { version = "0.10" }
:dep argmin-math = { version = "0.4", features = ["vec"] }
use argmin::core::{CostFunction, Error, Executor, State};
use argmin::solver::neldermead::NelderMead;

// The "loss": a bowl with its lowest point at (3, -1).
struct Loss;
impl CostFunction for Loss {
    type Param = Vec<f64>;
    type Output = f64;
    fn cost(&self, p: &Vec<f64>) -> Result<f64, Error> {
        Ok((p[0] - 3.0).powi(2) + (p[1] + 1.0).powi(2))
    }
}
println!("loss defined (true minimum at (3, -1))");

Now we hand the problem to a solver. **Nelder-Mead** is a gradient-free method:
it walks a simplex (here a triangle of three starting points) downhill until it
converges — no derivatives required. The `Executor` runs the solver:

In [ ]:
{
    let solver = NelderMead::new(vec![
        vec![0.0, 0.0], vec![2.0, 0.0], vec![0.0, 2.0],
    ]);
    let result = Executor::new(Loss, solver)
        .configure(|state| state.max_iters(200))
        .run()
        .unwrap();
    let best = result.state().get_best_param().unwrap().clone();
    println!("found minimum at [{:.3}, {:.3}]", best[0], best[1]);
    println!("loss there  = {:.6}", result.state().get_best_cost());
}

The solver recovered `(3, -1)` with a loss of essentially zero — exactly what a
model's `.fit()` does when it minimizes *its* loss over *its* parameters.
`argmin` also offers gradient-based methods (steepest descent, L-BFGS, etc.) for
when you *can* compute derivatives, which converge faster on smooth losses.

Next: [hyperparameter search](hyperparameter-search.ipynb) — optimizing the
values you choose *before* training, where the objective (a cross-validated
score) is noisy and has no gradient at all.